`Qwen/Qwen3-0.6B`: post-trained instruction-following checkpoint, with non-thinking chat formatting. 

In [1]:
%pip install "transformers==5.17.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 90.0 MB/s eta 0:00:00:00:010:01
  Attempting uninstall: transformers
    Found existing installation: transformers 5.16.1
    Uninstalling transformers-5.16.1:
      Successfully uninstalled transformers-5.16.1


In [2]:
import platform
import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("CUDA:", torch.version.cuda)
assert torch.cuda.is_available(), "Select a GPU runtime in Colab before continuing."
print("GPU:", torch.cuda.get_device_name(0))

Python: 3.13.15
PyTorch: 2.11.0+cu128
Transformers: 5.17.0
CUDA: 12.8
GPU: Tesla T4


In [3]:
model_id = "Qwen/Qwen3-0.6B"
# fixes the HF checkpoint and tokenizer
model_revision = "c1899de289a04d12100db370d81485cdf75e47ca"
device = torch.device("cuda")
dtype = torch.float32

In [4]:
tokenizer = AutoTokenizer.from_pretrained(
    model_id,
    revision=model_revision,
)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

In [5]:
# load model from HF
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    revision=model_revision,
    dtype=dtype,
    # non optimized, PyTorhc attention
    attn_implementation="eager",
).to(device)
# no grad retention
model.eval()
model.requires_grad_(False)


print(type(model).__name__)
print("Blocks:", model.config.num_hidden_layers)
print("Residual width:", model.config.hidden_size)
print("Vocabulary dimension:", model.config.vocab_size)

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.50GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Qwen3ForCausalLM
Blocks: 28
Residual width: 1024
Vocabulary dimension: 151936


In [6]:
# run a single forward pass
def forward_tokens(model : AutoModelForCausalLM, token_ids):
    # embed the token IDs: [B, T] -> [B, T, d_model]
    embeddings = model.get_input_embeddings()(token_ids)
    # inference: [B, T, d_model] -> [B, T, VocabSize]
    outputs = model(
        inputs_embeds=embeddings,
        use_cache=False,
        logits_to_keep=0,
    )
    return token_ids, embeddings, outputs.logits

# wrapper for string prompts
def read_prompt(model : AutoModelForCausalLM, text):
    # tokenize according to loaded tokenizer
    token_ids = tokenizer(
        text,
        return_tensors="pt",
        add_special_tokens=False,
    ).input_ids.to(device)
    if token_ids.shape[1] == 0:
        raise ValueError("The text must contain at least one token.")
    # pass tokens [B, T] to forward pass -> logits [B, T, VocabSize]
    return forward_tokens(model, token_ids)

In [7]:
question = "What is the capital of France? Answer with just the city."
prompt_text = tokenizer.apply_chat_template(
    [{"role": "user", "content": question}],
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False,
)
print(repr(prompt_text))

'<|im_start|>user\nWhat is the capital of France? Answer with just the city.<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n'


In [8]:
# run the forward pass
with torch.no_grad():
    token_ids, embeddings, logits = read_prompt(model, prompt_text)

print("token_ids [batch, position]:", tuple(token_ids.shape))
print("embeddings [batch, position, coordinate]:", tuple(embeddings.shape))
print("logits [batch, position, vocabulary]:", tuple(logits.shape))
print("IDs:", token_ids[0].tolist())
print("Pieces:", tokenizer.convert_ids_to_tokens(token_ids[0].tolist()))

token_ids [batch, position]: (1, 25)
embeddings [batch, position, coordinate]: (1, 25, 1024)
logits [batch, position, vocabulary]: (1, 25, 151936)
IDs: [151644, 872, 198, 3838, 374, 279, 6722, 315, 9625, 30, 21806, 448, 1101, 279, 3283, 13, 151645, 198, 151644, 77091, 198, 151667, 271, 151668, 271]
Pieces: ['<|im_start|>', 'user', 'Ċ', 'What', 'Ġis', 'Ġthe', 'Ġcapital', 'Ġof', 'ĠFrance', '?', 'ĠAnswer', 'Ġwith', 'Ġjust', 'Ġthe', 'Ġcity', '.', '<|im_end|>', 'Ċ', '<|im_start|>', 'assistant', 'Ċ', '<think>', 'ĊĊ', '</think>', 'ĊĊ']


In [9]:
# compute softmax with temperature
def temperature_softmax(logits, temperature=1.0):
    if not temperature > 0:
        raise ValueError("Temperature must be positive.")
    return torch.softmax(logits / temperature, dim=-1)

In [10]:
# set temp
temperature = 0.7
# extract last logit vector [1, VocabSize]
next_logits = logits[0, -1, :]
# -> probabilities
next_probabilities = temperature_softmax(next_logits, temperature)
# pull top 10
top_probabilities, top_ids = next_probabilities.topk(10)

print("Probability sum:", next_probabilities.sum().item())
for token_id, probability in zip(top_ids.tolist(), top_probabilities.tolist()):
    print(token_id, repr(tokenizer.decode([token_id])), probability)

Probability sum: 1.0000001192092896
59604 'Paris' 0.9340003132820129
43 'L' 0.037702690809965134
33 'B' 0.012225081212818623
44 'M' 0.005111338570713997
51 'T' 0.00236741011030972
49 'R' 0.001992483390495181
2304 'Le' 0.001575333415530622
2580 'Str' 0.0010509546846151352
1143 'Ch' 0.0007078003254719079
6828 'Br' 0.0006569097167812288


In [11]:
# take in tokens [B, T] -> [B, T + max_new_tokens]
def sample_tokens(model : AutoModelForCausalLM, token_ids, max_new_tokens=20, temperature=0.7):
    if max_new_tokens < 0:
        raise ValueError("max_new_tokens must be nonnegative.")
    # copy tokens locally (will be appended-to)
    sequence = token_ids.clone()
    # get the config-specified index for EOS token (to see when model's done)
    eos_token_ids = model.generation_config.eos_token_id
    for step in range(max_new_tokens):
        # get a single forward pass's logits
        _, _, step_logits = forward_tokens(model, sequence)
        # -> probabilities
        probabilities = temperature_softmax(step_logits[:, -1, :], temperature)
        # sample the distribution
        next_token = torch.multinomial(probabilities, num_samples=1)
        # append sampled token ID to sequence for next pass
        sequence = torch.cat((sequence, next_token), dim=1)
        if next_token.item() in eos_token_ids:
            break
    return sequence

In [12]:
torch.manual_seed(0)
with torch.no_grad():
    # call autoregressive sampling
    sequence = sample_tokens(model, token_ids, max_new_tokens=20, temperature=0.7)

# what are the new token IDs
new_ids = sequence[0, token_ids.shape[1]:]
print("New IDs:", new_ids.tolist())
# literal decoded tokens
print("Literal continuation:", repr(tokenizer.decode(new_ids.tolist())))
# human-readable decoded tokens
print("Answer:", tokenizer.decode(new_ids.tolist(), skip_special_tokens=True))

New IDs: [59604, 13, 151645]
Literal continuation: 'Paris.<|im_end|>'
Answer: Paris.


# Extract residual activations at specified layers and positions

In [13]:
def activations(model : AutoModelForCausalLM, token_ids, positions=None, layers=None):
    if token_ids.ndim != 2 or token_ids.shape[0] != 1:
        raise ValueError(f"Expected token_ids with shape [1, T]. Got {token_ids.shape}")

    # residual boundaries after each layer. boundary 0 gives you embedded tokens
    boundaries = [model.get_input_embeddings(), *model.model.layers]

    # none for either positions or layers gives you all
    if positions is None:
        positions = range(token_ids.shape[1])
    if layers is None:
        layers = range(len(boundaries))

    positions = list(positions)
    layers = list(layers)

    if not positions or not layers:
        raise ValueError("Must select at least one position or layer or None for all")

    # {layer : tensor}
    captured = {}
    # references to hooks we've attached
    handles = []

    # wrapper 
    def make_hook(layer):
        # hook takes a module from the torch model, its args, and then the output of the module
        def hook(module, args, output):
            # we save the specified positions of this layer's output to captured
            captured[layer] = output[0, positions, :]
        return hook

    try:
        for layer in layers:
            # reference to the hook we've attached to the model so we can remove it after
            handle = boundaries[layer].register_forward_hook(make_hook(layer))
            handles.append(handle)
        with torch.no_grad():
            # with hooks attached, run forward pass which will populate captured{}
            forward_tokens(model, token_ids)
    finally:
        # detach hooks from model's modules
        for handle in handles:
            handle.remove()

    # dim = 0 --> [layer, position, residual]
    return torch.stack([captured[layer] for layer in layers])

# Sanity check: unembed(activations(..., [-1], [-1])) = logits

In [14]:
unembed = model.get_output_embeddings()

with torch.no_grad():
    # [batch, positions, vocabulary]
    _, _, logits = forward_tokens(model, token_ids)
    # [vocabulary]
    final_logit = logits[0, -1, :]

    # [residual]
    final_latent = activations(model, token_ids, [-1], [-1])[0, 0]
    normalized_latent = model.model.norm(final_latent)
    # [vocabulary]
    reconstructed_logit = unembed(normalized_latent)

for label, logit in [("Full forward pass", final_logit), ("Reconstructed via final latent", reconstructed_logit)]:
    probabilities = temperature_softmax(logit, 0.1)
    top_probabilities, top_ids = probabilities.topk(10)
    print(f"\n{label}:")
    for token_id, probability in zip(top_ids.tolist(), top_probabilities.tolist()):
        print("\t", token_id, repr(tokenizer.decode([token_id])), probability)


Full forward pass:
	 59604 'Paris' 1.0
	 43 'L' 1.7465102974956181e-10
	 33 'B' 6.58170011005009e-14
	 44 'M' 1.469975652076391e-16
	 51 'T' 6.721872316855978e-19
	 49 'R' 2.0106183506568117e-19
	 2304 'Le' 3.88309889611046e-20
	 2580 'Str' 2.2838089166516134e-21
	 1143 'Ch' 1.4353167660492714e-22
	 6828 'Br' 8.513578740117309e-23

Reconstructed via final latent:
	 59604 'Paris' 1.0
	 43 'L' 1.7465637269786782e-10
	 33 'B' 6.580997411517048e-14
	 44 'M' 1.469818553934455e-16
	 51 'T' 6.721257101275392e-19
	 49 'R' 2.0105263268136652e-19
	 2304 'Le' 3.8828620510366316e-20
	 2580 'Str' 2.2837392444564642e-21
	 1143 'Ch' 1.4352510066042122e-22
	 6828 'Br' 8.512929349820131e-23


In [16]:
# Run a latent from a layer boundary through to the final residual tensor -> [1, T, 1024]
def R(model, latent, boundary, kwargs):
    for block in model.model.layers[boundary:]:
        latent = block(latent, **kwargs)
    return latent

# Return the final residual vector at index 'target' from R() -> [1024]
def R_target(model, latent, boundary, target, kwargs):
    return R(model, latent, boundary, kwargs)[0, target, :]

# Return the final residual vectors summed over positions -> [1024]
def R_summed(model, latent, boundary, kwargs):
    return R(model, latent, boundary, kwargs)[0].sum(dim=0)


In [ ]:
# now run a forward pass with some hooks and verify that these things all line up with identities. 

# run forward pass, sum hooked final residuals

# verify that R(., ., l-1, .) = model.model.layers[l] + R(., ., l, .)

In [19]:
l = 20 # output of layer 19, input for layer 20
captured = {}

# hook to attach to layers
def capture_entry(module, args, kwargs, output):
    captured["h_l"] = args[0] # the tensor passed into block l: [1, T, 1024]
    captured["kwargs"] = kwargs # anything else passed to the module

# hook to capture final layer latent
def capture_final(module, args, output):
    captured["h_L"] = output # [1, T, 1024]

handles = [
    model.model.layers[l].register_forward_hook(capture_entry, with_kwargs=True),
    model.model.layers[-1].register_forward_hook(capture_final)
]

# run the hooked forward pass
try:
    with torch.no_grad():
        _, embeddings, _ = forward_tokens(model, token_ids)
# rinse the handles
finally:
    for handle in handles:
        handle.remove()

h_l, h_l_kwargs, h_L = captured["h_l"], captured["kwargs"], captured["h_L"]

print(h_L - R(model, h_l, l, h_l_kwargs))

tensor([[[0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         ...,
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.]]], device='cuda:0')
